# Calculate a coverage area based on existing nodes

Unlike an ISP that provides the internet through phone lines, coaxial cable or fiber optic cable, Tucson Mesh's coverage area is a little more nuanced.

One way to approximate an area is based on where we have successful installs already.

This notebook calculates a bounding box around existing installs. Using hexbins could give a more granular sense of the coverage area, but a bounding box is easy and computationally lightweight to determine if coordinates fall in the box. This is a key use case for the coverage area - to process submissions of the interest form and put them in a queue based on the potential for getting a good connection.

A future improvement could be to incorporate other successful signal strength tests beyond installs.

In [1]:
import io
import json

from IPython.display import display
import ipywidgets as widgets
from lonboard import viz
import polars as pl
import polars_st as st

In [2]:
# TODO: Add an event handler so the rest of the code runs when a file is uploaded, rather than having to execute this notebook step by step.
uploader = widgets.FileUpload(
    accept=".geojson",
    multiple=False
)

In [3]:
uploader

FileUpload(value=(), accept='.geojson', description='Upload')

In [5]:
nodes = st.read_file(io.BytesIO(uploader.value[0].content))

In [6]:
nodes_installed = nodes.filter(pl.col("status").eq("Installed"))

In [7]:
installed_bounds = nodes_installed.select(
    st.rectangle(
        pl.col("geometry").st.total_bounds()
    ).st.set_srid(4326)
).with_columns(pl.lit("installed node bounds").alias("label"))

In [10]:
tucson_house = pl.DataFrame({
    "coords": [     
        [-110.979158, 32.240356],        
    ]
}).select(
    # Convert coordinates to a geometry
    st.point("coords")
    # Set a spatial reference system, otherwise we can't convert to others
    .st.set_srid(4326)
    .alias("geometry")
)

In [11]:
ft_in_mi = 5280

In [13]:
# Also create a geometry of 1 mile around the Tucson House supernode,
# just to see how our actual service area compares
tucson_house_1mi_buffer = tucson_house.select(
    pl.col("geometry")
        # First convert to a feet-based spatial reference system,
        # NAD83 / Arizona Central (ft).
        # See https://spatialreference.org/ref/epsg/2223/.
        .st.to_srid(2223)
        # Add one mile (in feet)
        .st.buffer(ft_in_mi)
        # Convert back to WSG 84
        .st.to_srid(4326)
).with_columns(
    pl.lit("1 mile buffer around Tucson House").alias("label")
)

In [14]:
all_bounds = pl.concat([
    installed_bounds,
    tucson_house_1mi_buffer,
])

## A bounding box of installed nodes, and a one mile buffer around the supernode

In [15]:
viz(all_bounds.st)

/home/ghing/workspace/tucsonmesh/tucsonmesh-scripts/analysis/.venv/lib/python3.13/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


## This is a bounding box for the installed nodes

In [16]:
# Show the bounding box coordinates
nodes_installed.select(
    pl.col("geometry").st.total_bounds(),
).item(0, "geometry").to_list()

[-111.0052761, 32.2240842, -110.9620514, 32.2543639]